In [ ]:
! pip install torchgeo

# TorchGeo Tutorial: Main Functionalities
## A comprehensive guide to TorchGeo for geospatial machine learning
TorchGeo is a PyTorch domain library for geospatial data that provides:
- Datasets for satellite imagery and geospatial data
- Samplers for efficient data loading
- Transforms for data augmentation
- Pre-trained models for transfer learning
- Utilities for working with geospatial data

## Download data from [here](https://drive.google.com/file/d/1cs14AgQw0czqFLOn_N2m40tj6ZlZhwLi/view?usp=sharing) , unzip and put sentinel-2 and labels folders in data/


In [2]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

from torchgeo.datasets import RasterDataset
from torchgeo.samplers import RandomGeoSampler, GridGeoSampler, Units



## 1. DATASETS: Built-in Geospatial Datasets

TorchGeo provides two main types of datasets:
- NonGeoDataset: Image classification datasets (like ImageNet)
- GeoDataset: Datasets with geospatial metadata (coordinates, CRS)


In [ ]:

print("\n" + "=" * 80)
print("1. DEFINE A GEOSPATIAL DATASET")
print("=" * 80)

# Example of a Geospatial Dataset with Sentinel-2 Images
class Sentinel2Dataset(RasterDataset):
    """
    Custom GeoDataset for Sentinel-2 imagery

    Sentinel-2 bands:
    - B02: Blue (490 nm)
    - B03: Green (560 nm)
    - B04: Red (665 nm)
    - B08: NIR (842 nm)
    - B11: SWIR1 (1610 nm)
    - B12: SWIR2 (2190 nm)
    """

    # File pattern to match your Sentinel-2 files
    # Adjust this based on your naming convention
    filename_glob = "*.tif"  # or "S2*.tif", "*_B*.tif", etc.
    filename_regex = r".*"

    # Specify if files are already in the correct format
    is_image = True

    # Separate files by timestamp if you have multi-temporal data
    separate_files = False

    # Band configuration (adjust based on your files)
    all_bands = ["B01","B02", "B03", "B04", "B05","B06", "B07", "B08", "B09", "B10", "B11", "B12"]
    rgb_bands = ["B04", "B03", "B02"]  # Red, Green, Blue for visualization


In [ ]:
# Initialize dataset
# Download zip from https://drive.google.com/file/d/1cs14AgQw0czqFLOn_N2m40tj6ZlZhwLi/view?usp=sharing extract here and update the path if needed
data_root = "data/sentinel-2"  # Update this path!

print(f"\nInitializing Sentinel-2 dataset from: {data_root}")
print("Make sure your directory structure looks like:")
print("  data/sentinel2/")
print("    ├── image1.tif")
print("    ├── image2.tif")
print("    └── ...")

# Create dataset
dataset = Sentinel2Dataset(paths=data_root)

print(f"\nDataset loaded!")
print(f"  Spatial bounds: {dataset.bounds}")
print(f"  CRS: {dataset.crs}")
print(f"  Resolution: {dataset.res}")


In [ ]:

# ============================================================================
# 2. RANDOM GEO SAMPLER: For Training
# ============================================================================

print("\n" + "=" * 80)
print("2. RANDOM GEO SAMPLER (Training)")
print("=" * 80)

# Random sampler configuration
patch_size = 128  # pixels
num_samples_per_epoch = 1000

# Create random sampler
random_sampler = RandomGeoSampler(
    dataset,
    size=patch_size,  # patch size in pixels
    length=num_samples_per_epoch,  # number of samples per epoch
    units=Units.PIXELS  # or Units.CRS for geographic units
)

print(f"\nRandom Sampler Configuration:")
print(f"  Patch size: {patch_size}x{patch_size} pixels")
print(f"  Samples per epoch: {num_samples_per_epoch}")
print(f"  Total possible area: {dataset.bounds}")

# Create DataLoader for training
def collate_fn(batch):
    """Custom collate function to stack samples"""
    return {
        'image': torch.stack([item['image'] for item in batch]),
        'bbox': [item['bounds'] for item in batch]
    }

train_dataloader = DataLoader(
    dataset,
    batch_size=8,
    sampler=random_sampler,
    collate_fn=collate_fn,
    num_workers=2
)

print(f"\nTraining DataLoader created:")
print(f"  Batch size: {train_dataloader.batch_size}")
print(f"  Batches per epoch: {len(train_dataloader)}")

In [ ]:

# ============================================================================
# 3. GRID GEO SAMPLER: For Inference
# ============================================================================

print("\n" + "=" * 80)
print("3. GRID GEO SAMPLER (Inference)")
print("=" * 80)

# Grid sampler for systematic coverage
grid_patch_size = 128
stride = 128  # Non-overlapping: stride = patch_size; Overlapping: stride < patch_size

grid_sampler = GridGeoSampler(
    dataset,
    size=grid_patch_size,
    stride=stride,
    units=Units.PIXELS
)

print(f"\nGrid Sampler Configuration:")
print(f"  Patch size: {grid_patch_size}x{grid_patch_size} pixels")
print(f"  Stride: {stride} pixels")
print(f"  Overlap: {grid_patch_size - stride} pixels")
print(f"  Total patches: {len(grid_sampler)}")


# Create DataLoader for inference
inference_dataloader = DataLoader(
    dataset,
    batch_size=8,
    sampler=grid_sampler,
    collate_fn=collate_fn,
    num_workers=2
)

print(f"\nInference DataLoader created:")
print(f"  Batch size: 8")
print(f"  Total batches: {len(inference_dataloader)}")

In [ ]:
# ============================================================================
# 4. VISUALIZING SAMPLED PATCHES
# ============================================================================

print("\n" + "=" * 80)
print("4. VISUALIZING SAMPLES")
print("=" * 80)

# -----------------------------------------------------------------------------
# Utility: Normalize Sentinel-2 RGB for visualization
# -----------------------------------------------------------------------------
def normalize_rgb(image: torch.Tensor, percentile: float = 2) -> np.ndarray:
    """
    Normalize image for visualization using percentile stretching.

    Args:
        image: Tensor [C, H, W], typically Sentinel-2 patch.
        percentile: lower/upper percentile for contrast stretching.

    Returns:
        np.ndarray [H, W, 3] ready for plt.imshow().
    """
    # Select RGB bands: if >3 channels, assume [B4,B3,B2] = [R,G,B]
    if image.shape[0] >= 4:
        rgb = image[[3, 2, 1]].cpu().numpy()
    else:
        rgb = image[:3].cpu().numpy()

    # Flatten to compute percentiles safely
    p_low = np.percentile(rgb, percentile)
    p_high = np.percentile(rgb, 100 - percentile)

    if p_high == p_low:  # avoid division by zero
        p_high = p_low + 1e-6

    # Stretch and clip
    rgb = np.clip((rgb - p_low) / (p_high - p_low), 0, 1)

    return np.transpose(rgb, (1, 2, 0))  # CHW → HWC


# -----------------------------------------------------------------------------
# Visualization: Random Sampler
# -----------------------------------------------------------------------------
def visualize_random_samples(dataloader):
    """Visualize random samples from a dataloader (RGB only)."""
    batch = next(iter(dataloader))
    images = batch["image"]

    n_samples = min(len(images), 8)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Random Sampler: Training Patches", fontsize=16, fontweight="bold")

    for idx, ax in enumerate(axes.flat[:n_samples]):
        img = normalize_rgb(images[idx])
        ax.imshow(img)
        ax.set_title(f"Sample {idx + 1}", fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    return fig


# -----------------------------------------------------------------------------
# Visualization: Grid Sampler
# -----------------------------------------------------------------------------
def visualize_grid_samples(dataloader):
    """Visualize grid samples from a dataloader (systematic sampling)."""
    batch = next(iter(dataloader))
    images = batch["image"]

    n_samples = min(len(images), 8)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Grid Sampler: Systematic Coverage", fontsize=16, fontweight="bold")

    for idx, ax in enumerate(axes.flat[:n_samples]):
        img = normalize_rgb(images[idx])
        ax.imshow(img)
        ax.set_title(f"Grid Patch {idx + 1}", fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    return fig


# -----------------------------------------------------------------------------
# Visualization: Random samples with labels
# -----------------------------------------------------------------------------
def visualize_random_samples_with_labels(dataloader):
    """Visualize random image+mask pairs from dataloader."""
    batch = next(iter(dataloader))
    images = batch["image"]
    masks = batch["mask"]

    n_samples = min(len(images), 4)  # 4 samples → 8 subplots (image+mask)
    fig, axes = plt.subplots(n_samples, 2, figsize=(8, 3 * n_samples))
    fig.suptitle("Random Sampler: Image + Label Pairs", fontsize=16, fontweight="bold")

    for i in range(n_samples):
        img = normalize_rgb(images[i])
        mask = masks[i].squeeze().cpu().numpy()

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Image {i + 1}", fontsize=10)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title(f"Mask {i + 1}", fontsize=10)
        axes[i, 1].axis("off")

    plt.tight_layout()
    return fig
# Visualize samples
print("\nGenerating visualizations...")

try:
    fig1 = visualize_random_samples(train_dataloader)
    plt.show()

    fig2 = visualize_grid_samples(inference_dataloader)
    plt.show()

    print("✓ Visualizations complete!")
except Exception as e:
    print(f"Note: Visualization requires actual image files.")
    print(f"Error: {e}")


In [ ]:
# ============================================================================
# 5. EXAMPLE - COMBINING WITH LABELS (Intersection Dataset)
# ============================================================================

print("\n" + "=" * 80)
print("5. COMBINING IMAGERY WITH LABELS")
print("=" * 80)

# To combine Sentinel-2 imagery with labels:

from torchgeo.datasets import IntersectionDataset

# Create label dataset
class LabelDataset(RasterDataset):
    filename_glob = "*_DEL.tif"
    is_image = False  # This is a mask/label. The get_item will return a dict with "mask" entry

# Combine datasets
labels = LabelDataset(paths="data/labels")



print(f"Labels loaded!")
print(f"  Spatial bounds: {labels.bounds}")
print(f"  CRS: {labels.crs}")
print(f"  Resolution: {labels.res}")

print(f"labels.bounds == dataset.bounds: {labels.bounds == dataset.bounds}")

In [ ]:


# Only sample where both imagery and labels exist
combined = IntersectionDataset(dataset, labels)

def combined_collate_fn(batch):
    """Custom collate function to stack samples"""
    return {
        'image': torch.stack([item['image'] for item in batch]),
        'mask': torch.stack([item['mask'] for item in batch]),
        'bbox': [item['bounds'] for item in batch]
    }


# Use with sampler
sampler = RandomGeoSampler(combined, size=256, length=1000)
dataloader = DataLoader(combined, sampler=sampler, collate_fn=combined_collate_fn, batch_size=8)

print(f"\nRandom Sampler Configuration:")
print(f"  Patch size: {patch_size}x{patch_size} pixels")
print(f"  Samples per epoch: {num_samples_per_epoch}")
print(f"  Total possible area: {dataset.bounds}")

visualize_random_samples_with_labels(dataloader)

## TIPS

✓ Sampler Selection:
  - RandomGeoSampler: Training (diverse samples, data augmentation)
  - GridGeoSampler: Inference (complete coverage, reproducible)
  - Use overlapping stride (e.g., 128 for 256px patches) for smoother predictions

✓ Performance:
  - Set num_workers > 0 in DataLoader for parallel loading
  - Use larger batch sizes for Grid sampling (no randomness needed)
  - Cache small datasets in RAM with cache=True

✓ Data Preparation:
  - Ensure all GeoTIFFs have the same CRS and resolution
  - Use COG (Cloud Optimized GeoTIFF) format for faster loading
  - Organize by date for multi-temporal datasets

✓ Patch Size:
  - 256x256 or 512x512 common for semantic segmentation
  - Consider GPU memory when choosing patch size
  - Match patch size to your model's expected input

✓ Visualization:
  - Use percentile stretching for better RGB visualization
  - Check NDVI: (NIR - Red) / (NIR + Red) for vegetation
  - Visualize different band combinations (False color, etc.)